# Mejora 2: Reentrenamiento con Focal Loss

Focal Loss reduce la pérdida de los ejemplos fáciles (bien clasificados),
forzando al modelo a enfocarse en los difíciles — especialmente las clases
minoritarias como Infiltration.

**FL(pt) = -alpha_t × (1 - pt)^gamma × log(pt)**

Con gamma=2: si pt=0.9 (ejemplo fácil), el factor (1-0.9)²=0.01 reduce la
pérdida 100×. Si pt=0.1 (ejemplo difícil), (1-0.1)²=0.81 casi no la reduce.

**Tiempo estimado: ~60 min en T4.**
**Runtime: T4 GPU.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - necesitas T4'}")

H5_LOCAL      = '/content/nih_images.h5'
PROCESSED_DIR = '/content/processed_s2'
CLASS_NAMES   = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 2. Verificar HDF5 y CSVs

In [ ]:
import h5py, pandas as pd

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000):
    print('Copiando HDF5...')
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)

with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5: {len(hf["images"]):,} imágenes')

os.makedirs(PROCESSED_DIR, exist_ok=True)
for csv in ['train.csv', 'val.csv', 'test.csv']:
    local = f'{PROCESSED_DIR}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s2/{csv}', local)
    df = pd.read_csv(local)
    dist = {CLASS_NAMES[l]: int((df['label']==l).sum()) for l in range(4)}
    print(f'✓ {csv}: {len(df)} imgs | {dist}')


## 3. Instalar módulos de Focal Loss

In [ ]:
from pathlib import Path

# focal_loss.py
content = r'''"""
src/training/focal_loss.py
Focal Loss para clasificación multi-clase con clases desbalanceadas.

FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt)

  - gamma > 0: reduce la pérdida de ejemplos bien clasificados (fáciles),
    forzando al modelo a enfocarse en los difíciles (clases raras).
  - alpha: pesos por clase (igual que class_weights en CrossEntropy).

Valores típicos en imágenes médicas:
  gamma = 2.0 (valor original del paper RetinaNet, Lin et al. 2017)
  alpha = inversamente proporcional a la frecuencia de clase
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0,
                 alpha: torch.Tensor | None = None,
                 reduction: str = "mean"):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha      # tensor (num_classes,) o None
        self.reduction = reduction

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits:  (B, C) — logits crudos del modelo
            targets: (B,)   — labels enteros 0..C-1
        """
        # Probabilidades via softmax
        log_prob = F.log_softmax(logits, dim=1)          # (B, C)
        prob     = log_prob.exp()                         # (B, C)

        # Probabilidad de la clase verdadera
        # gather: selecciona prob[i, targets[i]] para cada i
        pt = prob.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)  # (B,)

        # Factor focal: (1 - pt)^gamma
        focal_weight = (1.0 - pt) ** self.gamma          # (B,)

        # Log-prob de la clase verdadera
        log_pt = log_prob.gather(
            dim=1, index=targets.unsqueeze(1)
        ).squeeze(1)                                      # (B,)

        # Loss base: -log(pt)
        loss = -focal_weight * log_pt                     # (B,)

        # Aplicar alpha si se proporcionó
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
            loss = alpha_t * loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss
'''
p = Path(f'{CODE_DIR}/src/training/focal_loss.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
# train_s2_fl.py
content = r'''"""
src/training/train_s2_fl.py
Loop de entrenamiento Sprint 2 con Focal Loss.
Basado en train_s2.py, con FocalLoss en lugar de CrossEntropyLoss.
"""
from __future__ import annotations
import copy
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from src.evaluation.metrics_s2 import (
    compute_metrics_multiclass, logits_to_probs_multiclass
)
from src.training.focal_loss import FocalLoss
from src.training.utils import save_checkpoint


def compute_alpha(train_loader, num_classes, device):
    """Alpha inversamente proporcional a la frecuencia de clase."""
    counts = torch.zeros(num_classes)
    for batch in train_loader:
        for label in batch["label"]:
            counts[label.item()] += 1
    total  = counts.sum()
    alpha  = total / (num_classes * counts)
    alpha  = alpha / alpha.sum() * num_classes
    return alpha.to(device)


def train_one_epoch_fl(model, loader, optimizer, criterion,
                       device, use_amp, scaler):
    model.train()
    losses, all_probs, all_labels = [], [], []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp and scaler:
            with autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        losses.append(loss.item())
        with torch.no_grad():
            probs = logits_to_probs_multiclass(logits.detach().float())
            all_probs.append(probs.cpu()); all_labels.append(labels.cpu())
    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return {"loss": float(np.mean(losses)),
            "metrics": compute_metrics_multiclass(all_labels, all_probs)}


@torch.no_grad()
def validate_one_epoch_fl(model, loader, criterion, device):
    model.eval()
    losses, all_probs, all_labels = [], [], []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        logits = model(images)
        losses.append(criterion(logits, labels).item())
        probs = logits_to_probs_multiclass(logits.float())
        all_probs.append(probs.cpu()); all_labels.append(labels.cpu())
    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return {"loss": float(np.mean(losses)),
            "metrics": compute_metrics_multiclass(all_labels, all_probs)}


def train_model_focal(model, train_loader, val_loader, cfg, device, logger):
    num_classes = cfg.model["num_classes"]
    trainable   = [p for p in model.parameters() if p.requires_grad]
    optimizer   = torch.optim.Adam(
        trainable, lr=cfg.training["learning_rate"],
        weight_decay=cfg.training.get("weight_decay", 0.0)
    )
    logger.info("Calculando alpha para Focal Loss...")
    alpha    = compute_alpha(train_loader, num_classes, device)
    logger.info("Alpha: %s", alpha.tolist())
    criterion = FocalLoss(gamma=2.0, alpha=alpha)
    logger.info("Focal Loss (gamma=2.0) activado.")

    use_amp  = cfg.training.get("use_amp", False) and torch.cuda.is_available()
    scaler   = GradScaler(device="cuda") if use_amp else None
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
    )

    epochs, patience = cfg.training["epochs"], cfg.training.get("patience", 8)
    ckpt_dir = Path(cfg.paths["checkpoints"]); ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = ckpt_dir / "sprint2_focal_best.pt"

    best_auc, best_epoch, best_state = -float("inf"), -1, None
    no_improve, history = 0, []

    logger.info("=" * 70)
    logger.info("Iniciando entrenamiento con Focal Loss: %d epochs", epochs)
    logger.info("=" * 70)

    for epoch in range(1, epochs + 1):
        lr = optimizer.param_groups[0]["lr"]
        tr = train_one_epoch_fl(model, train_loader, optimizer,
                                criterion, device, use_amp, scaler)
        va = validate_one_epoch_fl(model, val_loader, criterion, device)
        scheduler.step(va["loss"])

        logger.info(
            "Epoch %2d/%d | LR=%.2e | train loss=%.4f AUC=%.4f | "
            "val loss=%.4f AUC=%.4f acc=%.4f",
            epoch, epochs, lr,
            tr["loss"], tr["metrics"]["auc_macro"],
            va["loss"], va["metrics"]["auc_macro"], va["metrics"]["accuracy"],
        )
        for i, auc in enumerate(va["metrics"]["auc_per_class"]):
            logger.info("  val AUC clase %d: %.4f", i, auc)

        history.append({
            "epoch": epoch, "lr": lr,
            "train_loss": tr["loss"], "train_metrics": tr["metrics"],
            "val_loss":   va["loss"], "val_metrics":   va["metrics"],
        })

        score = va["metrics"]["auc_macro"]
        if score > best_auc:
            best_auc, best_epoch = score, epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            logger.info("  → Nuevo mejor (AUC_macro=%.4f)", best_auc)
        else:
            no_improve += 1
            if no_improve >= patience:
                logger.info("Early stopping.")
                break

    if best_state:
        save_checkpoint({
            "epoch": best_epoch, "model_state_dict": best_state,
            "best_val_auc_macro": best_auc,
            "loss": "FocalLoss", "gamma": 2.0,
            "cfg_snapshot": {
                "backbone": cfg.model["backbone"],
                "num_classes": cfg.model["num_classes"],
                "finetune_mode": cfg.model["finetune_mode"],
            },
        }, best_ckpt)
        logger.info("Mejor modelo guardado: %s (epoch %d, AUC=%.4f)",
                    best_ckpt, best_epoch, best_auc)

    logger.info("Entrenamiento Focal Loss terminado.")
    return {"history": history, "best_epoch": best_epoch,
            "best_val_auc_macro": best_auc, "checkpoint_path": str(best_ckpt)}
'''
p = Path(f'{CODE_DIR}/src/training/train_s2_fl.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
# run_train_s2_focal.py
content = r'''"""
scripts/run_train_s2_focal.py
Entrypoint Sprint 2 con Focal Loss.
"""
from __future__ import annotations
import json, os, sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_hdf5 import build_nih_hdf5_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.baseline import build_baseline_model
from src.training.train_s2_fl import train_model_focal
from src.training.utils import get_device, get_logger, load_config, set_seed


def main():
    h5_path       = os.environ.get("H5_PATH",       "/content/nih_images.h5")
    processed_dir = os.environ.get("PROCESSED_DIR", "/content/processed_s2")
    cfg    = load_config(CODE_DIR / "configs" / "sprint2.yaml")
    set_seed(cfg.project["seed"])
    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("train_s2_focal",
                        log_file=Path(cfg.paths["logs"]) / "train_s2_focal.log")
    logger.info("=" * 70)
    logger.info("RUN TRAIN S2 — FOCAL LOSS | 4 clases | finetune full")
    logger.info("=" * 70)
    device = get_device()
    logger.info("Device: %s%s", device,
                f" | {torch.cuda.get_device_name(0)}" if device.type=="cuda" else "")
    train_tfm = build_train_transform(cfg.preprocessing["image_size"])
    eval_tfm  = build_eval_transform(cfg.preprocessing["image_size"])
    train_ds, val_ds, _ = build_nih_hdf5_datasets(
        processed_dir, h5_path, train_tfm, eval_tfm)
    logger.info("Train: %d | Val: %d", len(train_ds), len(val_ds))
    nw = min(cfg.training.get("num_workers", 2), 4)
    train_loader = DataLoader(train_ds, batch_size=cfg.training["batch_size"],
                              shuffle=True, num_workers=nw,
                              pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=cfg.training["batch_size"],
                              shuffle=False, num_workers=nw,
                              pin_memory=(device.type=="cuda"))
    model = build_baseline_model(cfg).to(device)
    logger.info("Modelo: %s | mode=%s | trainable=%d",
                model.__class__.__name__, cfg.model["finetune_mode"],
                model.trainable_params())
    result = train_model_focal(model, train_loader, val_loader, cfg, device, logger)
    hp = Path(cfg.paths["experiments"]) / "history_s2_focal.json"
    hp.parent.mkdir(parents=True, exist_ok=True)
    with hp.open("w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    logger.info("History: %s", hp)
    logger.info("RESUMEN: best epoch=%d | AUC_macro=%.4f",
                result["best_epoch"], result["best_val_auc_macro"])


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/run_train_s2_focal.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


## 4. Entrenar con Focal Loss (~60 min)

In [ ]:
import subprocess

env = {**os.environ, 'H5_PATH': H5_LOCAL, 'PROCESSED_DIR': PROCESSED_DIR}

print('Iniciando entrenamiento con Focal Loss...')
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train_s2_focal.py'],
    capture_output=True, text=True, env=env,
)
print('LOG:')
lines = result.stderr.strip().split('\n') if result.stderr else []
print('\n'.join(lines[-80:]))
print(f'\nReturn code: {result.returncode}')


## 5. Comparativa: CrossEntropy vs Focal Loss

In [ ]:
import json, matplotlib.pyplot as plt
import numpy as np

# Cargar historiales
with open(f'{PROJECT_ROOT}/experiments_s2/history_s2.json') as f:
    ce_data = json.load(f)
with open(f'{PROJECT_ROOT}/experiments_s2/history_s2_focal.json') as f:
    fl_data = json.load(f)

ce_hist = ce_data['history']
fl_hist = fl_data['history']

print('COMPARATIVA FINAL (val set):')
print('=' * 65)
print(f"{'Métrica':<20} {'CrossEntropy':>15} {'Focal Loss':>12} {'Delta':>10}")
print('-' * 65)

ce_best = ce_hist[ce_data['best_epoch']-1]['val_metrics']
fl_best = fl_hist[fl_data['best_epoch']-1]['val_metrics']

for key, label in [
    ('auc_macro',         'AUC macro'),
    ('accuracy',          'Accuracy'),
    ('sensitivity_macro', 'Sens macro'),
    ('specificity_macro', 'Spec macro'),
]:
    v1, v2 = ce_best[key], fl_best[key]
    arrow  = '↑' if v2-v1 > 0.01 else ('↓' if v2-v1 < -0.01 else '≈')
    print(f"{label:<20} {v1:>15.4f} {v2:>12.4f} {v2-v1:>+10.4f} {arrow}")

print()
print(f"{'Clase':<15} {'CE AUC':>8} {'FL AUC':>8} {'CE Sens':>8} {'FL Sens':>8}")
print('-' * 50)
for i in range(4):
    ce_auc  = ce_best['auc_per_class'][i]
    fl_auc  = fl_best['auc_per_class'][i]
    ce_sens = ce_best['sensitivity_per_class'][i]
    fl_sens = fl_best['sensitivity_per_class'][i]
    arrow   = '↑' if fl_auc-ce_auc > 0.01 else ('↓' if fl_auc-ce_auc < -0.01 else '≈')
    print(f"{CLASS_NAMES[i]:<15} {ce_auc:>8.4f} {fl_auc:>8.4f} "
          f"{ce_sens:>8.4f} {fl_sens:>8.4f} {arrow}")

# Plot comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3','#F44336','#4CAF50','#FF9800']

for ax, hist, label, style in [
    (axes[0], ce_hist, 'CrossEntropy', '-'),
    (axes[0], fl_hist, 'Focal Loss', '--'),
]:
    eps = [h['epoch'] for h in hist]
    aucs = [h['val_metrics']['auc_macro'] for h in hist]
    ax.plot(eps, aucs, style, label=f'{label} (best={max(aucs):.4f})')

axes[0].axhline(0.5, color='gray', linestyle=':', alpha=0.4)
axes[0].set_title('Val AUC Macro: CE vs FL')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC macro')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

# AUC por clase Focal Loss
for i in range(4):
    eps  = [h['epoch'] for h in fl_hist]
    aucs = [h['val_metrics']['auc_per_class'][i] for h in fl_hist]
    axes[1].plot(eps, aucs, 'o-', ms=3, label=CLASS_NAMES[i], color=colors[i])

axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.4)
axes[1].set_title('Val AUC por clase — Focal Loss')
axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

plt.suptitle(f"Focal Loss: AUC_macro={fl_data['best_val_auc_macro']:.4f} @ epoch {fl_data['best_epoch']}")
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/focal_vs_ce.png',
            dpi=110, bbox_inches='tight')
plt.show()
print('✓ Guardado.')


## 6. Siguiente paso

Si Focal Loss mejoró el AUC macro → evaluar sobre test con el nuevo checkpoint.
Vuelve al chat para el notebook de cierre final del Sprint 2.
